# Diagnosis-guided regeneration

Round 1 → diagnostics → **Stage A** (decisions) → **Stage B** (one model per branch) → union pool.

In [5]:
import sys, json, textwrap
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from ssc import regenerate as R
from ssc.prompts_regen import decision_messages, branch_messages

BENCH, PID, ARM, LIVE = "mamo_complex", "mamo_complex_0001", "branch", False
load = lambda *p: json.load(open(ROOT.joinpath("runs", *p)))
pool, score = load(BENCH, "pools", f"{PID}.json"), load(BENCH, "scores", f"{PID}.json")
ev1, reqs = load(BENCH, "eval", f"{PID}.json"), score["requirements"]

print(textwrap.fill(pool["question"], 110)[:1500], "...\n")
print(f"Gold Answer: {ev1['gold']}\n\nRound 1 (K = 10):")
print(f"{'cand':>4} {'SSC':>6} {'LP obj':>9} {'MILP obj':>9}  correct")
for i, (c, e) in enumerate(zip(score["candidates"], ev1["candidates"])):
    sel = "  <- SSC pick" if i == score["selected"]["SSC"] else ""
    print(f"{i:>4} {c['SSC']:6.3f} {str(round(c['lp_objective'],3)):>9} {str(round(e['objective'] or 0,3)):>9}  {e['correct']}{sel}")
print("\nCoverage:", any(e["correct"] for e in ev1["candidates"]))

Imagine you're a dietitian, tasked with planning a nutritious and balanced meal for an athlete. The athlete
needs to maintain a high protein, moderate carb diet, with enough calories to sustain their intense training
regimen. The goal is to meet these dietary requirements at the lowest cost possible. You have five different
food items to choose from: Steak, Rice, Chicken, Iceberg Lettuce, and Almonds. Each of these foods contains
varying amounts of protein, carbohydrates, and calories, and each comes with a specific cost.  Here's a
detailed breakdown of the nutritional value of each food item and its cost:  - Steak: Offers a whopping 20
grams of protein, 12 grams of carbohydrates, and 167 calories for $7. - Rice: Provides 6 grams of protein, an
impressive 29 grams of carbohydrates, and a hefty 275 calories for $8. - Chicken: Packs 17 grams of protein,
26 grams of carbohydrates, and 112 calories for only $2. - Iceberg Lettuce: Contains only 1 gram of protein, 8
grams of carbohydrates, a

In [6]:
# Diagnostics (gold-free) and the Stage A prompt
slots = R.build_slots(pool, score, reqs)
print("Risk features:", {k: round(v, 3) for k, v in slots["risk"].items()})
print("Trigger fires:", R.triggered(slots["risk"]), "\n")
for k in ("pool_summary", "signature_diff", "diagnostics"):
    print(f"--- {k}\n{slots[k]}\n")

msgs_A = decision_messages(pool["question"], slots)
print("=== Stage A user prompt (task part) ===")
print(msgs_A[1]["content"].split("## Task")[1][:1200])

if LIVE:
    from ssc.llm import LLMClient
    llm = LLMClient(model="deepseek-flash")
    raw = llm.chat(msgs_A, temperature=0.0, max_tokens=2000, tag=f"{PID}:nbA", json_mode=True)["content"]
    decisions = R.parse_decisions(raw)
else:
    decisions = load(f"{BENCH}_r2_{ARM}", "regen", f"{PID}.json")["decisions"]

print("\n=== Stage A output: decisions the text does not force ===")
for d in decisions:
    opts = ", ".join(f"{o} ({p:.2f})" for o, p in zip(d["options"], d["plausibility"]))
    print(f"- {d['decision']}: {opts}   | round-1 majority: {d['majority_option']}")

Risk features: {'top_lp_cluster_mass': 1.0, 'largest_lp_cluster': 1.0, 'n_lp_clusters': 1, 'frac_lp_failed': 0.0, 'frac_fatal': 0.0, 'max_SSC': 0.918, 'ssc_margin': 0.0, 'top_TA': 0.75, 'top_SQ': 1.0}
Trigger fires: False 

--- pool_summary
10 models were built.
- Group A: 10 model(s), LP-relaxation objective ~ 17.3977

--- signature_diff
- Only one group of models exists; no structural contrast is available.

--- diagnostics
- A resource-availability limit stated in the text may be missing. (requirement 'food_selection'; present in 0% of models)
- 1 model(s) may multiply or divide decision variables; the model must be linear.

=== Stage A user prompt (task part) ===

List every modelling decision that the problem text does NOT force uniquely. Always consider:
  - the domain of each decision variable (Binary / NonNegativeIntegers / NonNegativeReals / Reals),
  - the objective sense and what exactly is being optimised,
  - the direction (<=, >=, =) and inclusiveness of each limit,
  - t

In [7]:
# Choose branches (top-B by plausibility, majority excluded) and the Stage B prompt
assignments = R.top_assignments(decisions, B=4)
for b, a in enumerate(assignments):
    changed = [f"{x['decision']} = {x['option']}" for x, d in zip(a, decisions) if x["option"] != d["majority_option"]]
    print(f"branch {b}: differs from round-1 majority in -> {changed}")

msgs_B = branch_messages(pool["question"], assignments[0])
print("\n=== Stage B user prompt for branch 0 (decisions part) ===")
print(msgs_B[1]["content"].split("## Modelling decisions")[1].split("## Task")[0])

new_pool = load(f"{BENCH}_r2_{ARM}", "pools", f"{PID}.json")
code0 = new_pool["candidates"][0]["code"]
print("=== Generated model, branch 0 (variables + assumptions) ===")
print("\n".join(l for l in code0.splitlines() if "Var(" in l or "assumption" in l.lower() or "sense=" in l))

branch 0: differs from round-1 majority in -> ['domain_of_food_servings = NonNegativeIntegers']
branch 1: differs from round-1 majority in -> ['food_selection_limit = at most one serving per food (binary selection)']
branch 2: differs from round-1 majority in -> ['domain_of_food_servings = NonNegativeIntegers', 'food_selection_limit = at most one serving per food (binary selection)']
branch 3: differs from round-1 majority in -> ['food_selection_limit = at most one of each food type selected']

=== Stage B user prompt for branch 0 (decisions part) ===
 to follow (fixed; do not revisit them)
- domain_of_food_servings: NonNegativeIntegers   (text silent; treat as an assumption)
- objective_sense: minimize total cost   (source: "The goal is to meet these dietary requirements at the lowest cost possible.")
- nutrient_constraint_direction: >= requirement for each nutrient   (source: "at least 83 grams of protein, 128 grams of carbohydrates, and 1686 calories")
- food_selection_limit: no add

In [8]:
# Result: round 1 vs union pool (10 + 4)
ev2 = load(f"{BENCH}_r2_{ARM}", "eval", f"{PID}.json")
print("New candidates (SCIP, offline evaluation):")
for b, e in enumerate(ev2["candidates"]):
    print(f"  branch {b}: status={e['status']:<10} objective={e['objective']}  correct={e['correct']}")

su = load(f"{BENCH}_r2_{ARM}_union", "scores", f"{PID}.json")
evu = load(f"{BENCH}_r2_{ARM}_union", "eval", f"{PID}.json")
c1 = su["selected"]["SSC"]
print(f"\nCoverage   round 1: {any(e['correct'] for e in ev1['candidates'])}   ->  union: {any(e['correct'] for e in evu['candidates'])}")
print(f"SSC pick on union: candidate {c1} (round {1 if c1 < 10 else 2}), correct = {evu['candidates'][c1]['correct']}")

sys.path.insert(0, str(ROOT / "scripts"))
from eval_top2 import load_benchmark, pick_hybrid
p = next(p for p in load_benchmark(str(ROOT), f"{BENCH}_r2_{ARM}_union", "scores") if p["pid"] == PID)
c2 = pick_hybrid(p, p["c1"], lam=0.1, delta=1e-3)
print(f"Hybrid 2nd pick: candidate {c2} (round {1 if c2 < 10 else 2}), correct = {evu['candidates'][c2]['correct']}")

New candidates (SCIP, offline evaluation):
  branch 0: status=optimal    objective=17.999999999999996  correct=True
  branch 1: status=infeasible objective=None  correct=False
  branch 2: status=infeasible objective=None  correct=False
  branch 3: status=infeasible objective=None  correct=False

Coverage   round 1: False   ->  union: True
SSC pick on union: candidate 7 (round 1), correct = False
Hybrid 2nd pick: candidate 10 (round 2), correct = True


In [9]:
# Variables chosen: round-1 SSC pick vs Hybrid pick on the union pool (offline SCIP solve)
import os, sys, tempfile
from pyscipopt import Model as ScipModel
sys.path.insert(0, str(ROOT / "scripts"))
from eval_top2 import load_benchmark, pick_hybrid

def nonzero_vars(code):
    ns = {"__name__": "candidate"}; exec(code, ns); m = ns["build_model"]()
    fd, lp = tempfile.mkstemp(suffix=".lp"); os.close(fd)
    m.write(lp, io_options={"symbolic_solver_labels": True})
    s = ScipModel(); s.hideOutput(); s.readProblem(lp); s.optimize(); os.remove(lp)
    if s.getStatus() != "optimal":
        return s.getStatus()
    xs = ", ".join(f"{v.name.replace('(', '[').replace(')', ']')}={s.getVal(v):.3g}"
                   for v in s.getVars() if abs(s.getVal(v)) > 1e-9 and v.name != "ONE_VAR_CONSTANT")
    return f"{xs}  (obj {s.getObjVal():.4g})"

r1_codes = [c["code"] for c in load(BENCH, "pools", f"{PID}.json")["candidates"]]
u_codes = r1_codes + [c["code"] for c in load(f"{BENCH}_r2_{ARM}", "pools", f"{PID}.json")["candidates"]]
c1_r1 = load(BENCH, "scores", f"{PID}.json")["selected"]["SSC"]
p = next(p for p in load_benchmark(str(ROOT), f"{BENCH}_r2_{ARM}_union", "scores") if p["pid"] == PID)
c2_u = pick_hybrid(p, p["c1"], lam=0.1, delta=1e-3)

print(f"Round 1, SSC pick (cand {c1_r1}):       {nonzero_vars(r1_codes[c1_r1])}")
print(f"Union, Hybrid 2nd pick (cand {c2_u}):  {nonzero_vars(u_codes[c2_u])}")

Round 1, SSC pick (cand 2):       x[Chicken]=4.61, x[Almonds]=4.09  (obj 17.4)
Union, Hybrid 2nd pick (cand 10):  servings[Chicken]=5, servings[Almonds]=4  (obj 18)
